スペクトログラム＆フォルマント周波数の波形出力

In [4]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import lfilter
import soundfile as sf
import os

# ==========================================
# 1. 設定：比較したい2つの音声ファイル名
# ==========================================
files = {
    'harusu': 'nanishi_150_harusu_150_20260608_113724.wav',
    # 💡もう1つの「はるの」のファイル名（113650など）をここに正確に書いてください
    'haruno': 'nanishi_haruno_20260517_180128.wav' 
}

# フォルマント抽出用の内部関数（廣瀬さんのコードを移植）
def estimate_formants(y, sr):
    y_filt = lfilter([1, -0.63], [1], y)
    frame_length = int(0.025 * sr)
    hop_length = int(0.010 * sr)
    frames = librosa.util.frame(y_filt, frame_length=frame_length, hop_length=hop_length)
    f1_list, f2_list = [], []
    for frame in frames.T:
        windowed = frame * np.hamming(len(frame))
        a = librosa.lpc(windowed, order=18)
        roots = np.roots(a)
        roots = [r for r in roots if np.imag(r) >= 0]
        ang = np.arctan2(np.imag(roots), np.real(roots))
        frqs = sorted(ang * (sr / (2 * np.pi)))
        formants = [f for f in frqs if 250 < f < 3500]
        f1_list.append(formants[0] if len(formants) > 0 else 0)
        f2_list.append(formants[1] if len(formants) > 1 else 0)
    return np.array(f1_list), np.array(f2_list), hop_length

# ==========================================
# 2. メイン処理：位置合わせと全画像の一括生成
# ==========================================
for label, file_name in files.items():
    if not os.path.exists(file_name):
        print(f"【警告】ファイルが見つかりません: {file_name} (スキップします)")
        continue
        
    print(f"\n🎬 {file_name} の処理を開始します...")
    
    # ─── A. 音声の読み込みと自動位置合わせ ───
    # フォルマント計算の精度向上のため、16kHzリサンプリングで統一して読み込みます
    y_raw, sr = librosa.load(file_name, sr=16000)
    
    # 20dB以上の音（声の始まり）を検知して頭の無音を自動カット
    y_trimmed, index = librosa.effects.trim(y_raw, top_db=20)
    print(f" └ 前方の無音を {index[0]/sr:.2f} 秒カットして位置を揃えました。")
    
    # ─── B. 波形（Waveform）の生成と保存 ───
    plt.figure(figsize=(10, 3))
    librosa.display.waveshow(y_trimmed, sr=sr, color="blue")
    plt.title(f'Waveform - Aligned ({label})')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.tight_layout()
    
    wave_img_name = f'なにし_{label}.png'
    plt.savefig(wave_img_name)
    plt.close()
    print(f" └ 💾 保存完了: {wave_img_name}")
    
    # ─── C. スペクトログラム ＆ フォルマントの計算 ───
    D = librosa.stft(y_trimmed)
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    
    f1, f2, hop = estimate_formants(y_trimmed, sr)
    times = librosa.frames_to_time(range(len(f1)), sr=sr, hop_length=hop)
    
    # ─── D. フォルマント付スペクトログラムの生成と保存 ───
    plt.figure(figsize=(10, 5))
    librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz', cmap='magma')
    
    # フォルマント（F1:赤, F2:水色）を重ね書き
    plt.plot(times[f1>0], f1[f1>0], label='F1 (First Formant)', color='red', linewidth=2)
    plt.plot(times[f2>0], f2[f2>0], label='F2 (Second Formant)', color='cyan', linewidth=2)
    
    plt.ylim(0, 4000)  # 人間の声の主要帯域にズーム
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Spectrogram with Formants - Aligned ({label})')
    plt.xlabel('Time (s)')
    plt.ylabel('Frequency (Hz)')
    plt.legend(loc='upper right')
    plt.tight_layout()
    
    formant_img_name = f'なにし_{label}_F.png'
    plt.savefig(formant_img_name)
    plt.close()
    print(f" └ 💾 保存完了: {formant_img_name}")

print("\n✨ すべての画像が、スタート位置ピッタリの状態で保存されました！ ✨")


🎬 nanishi_150_harusu_150_20260608_113724.wav の処理を開始します...
 └ 前方の無音を 1.38 秒カットして位置を揃えました。
 └ 💾 保存完了: なにし_harusu.png
 └ 💾 保存完了: なにし_harusu_F.png

🎬 nanishi_haruno_20260517_180128.wav の処理を開始します...
 └ 前方の無音を 9.76 秒カットして位置を揃えました。
 └ 💾 保存完了: なにし_haruno.png
 └ 💾 保存完了: なにし_haruno_F.png

✨ すべての画像が、スタート位置ピッタリの状態で保存されました！ ✨
